# Oscar Romero López
# 2026-03-16

# Assignment 1: Data formats, Storage Optimization and MongoDB

In [1]:
import os
import time
import pyarrow as pa
import pyarrow.csv as pcsv
import pyarrow.parquet as pq
import duckdb
import pandas as pd
import urllib.request
import pyarrow.dataset as ds
import pyarrow.compute as pc

# 1. Getting to know the data (5%)
Explore the dataset:
- How many rows does each monthly file have? How many in total?
- What columns are available? What are their data types?
- Look at a few rows to understand the data
The TLC data dictionary is available at:
https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf

In [2]:
months = ["01", "02", "03"]
files = [f"yellow_tripdata_2024-{m}.parquet" for m in months]

In [3]:
metadata_storage = {} 

# 1. Download
for f in files:
    print(f"Downloading {f}...")
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{f}"
    urllib.request.urlretrieve(url, f)
    
# 2. Rows per file
for file in files:
    print(f"\n--- Metadata for {file} ---") 
    metadata = pq.read_metadata(file)
    metadata_storage[file] = metadata
    print(f"Number of rows: {metadata.num_rows:,}")

# 3. Whole dataset (quarter): total rows, columns and datatypes
merged = pa.concat_tables(
    [pq.read_table(f) for f in files], 
    promote_options='permissive'
)

output_path = "yellow_tripdata_2024_Q1.parquet"
pq.write_table(merged, output_path)

print(f"\n--- Resumen Q1 ---")
print(f"Filas totales: {merged.num_rows:,}")
print(f"Esquema:")
for field in merged.schema:
    print(f"  {field.name:25} | {field.type}")

# 4. Preview rows
print("\n--- Preview rows ---")
print(merged.slice(0, 3).to_pandas())


--- Metadata for yellow_tripdata_2024-01.parquet ---
Number of rows: 2,964,624

--- Metadata for yellow_tripdata_2024-02.parquet ---
Number of rows: 3,007,526

--- Metadata for yellow_tripdata_2024-03.parquet ---
Number of rows: 3,582,628

--- Resumen Q1 ---
Filas totales: 9,554,778
Esquema:
  VendorID                  | int32
  tpep_pickup_datetime      | timestamp[us]
  tpep_dropoff_datetime     | timestamp[us]
  passenger_count           | int64
  trip_distance             | double
  RatecodeID                | int64
  store_and_fwd_flag        | large_string
  PULocationID              | int32
  DOLocationID              | int32
  payment_type              | int64
  fare_amount               | double
  extra                     | double
  mta_tax                   | double
  tip_amount                | double
  tolls_amount              | double
  improvement_surcharge     | double
  total_amount              | double
  congestion_surcharge      | double
  Airport_fee             

# 2. MongoDB — Loading and Querying Semi-structured Data (20%)
In this part you'll load the taxi data into MongoDB and run some queries. This
connects what we learned in Session 2 (document databases) with the storage format
concepts from Session 3.

- 2.1. Load data into MongoDB
Start a MongoDB instance using Docker (you can reuse the compose.yaml from the Session 2 practice or create a new one).
Export a sample of 100,000 rows from the Parquet data as JSON Lines, and import it into MongoDB using `mongoimport`:
mongoimport --db taxi --collection trips --drop --file trips_sample.json



In [4]:
sample_rows = 100_000

sample = merged.slice(0, sample_rows)
sample_path = "trips_sample.json"

sample.to_pandas().to_json(
    sample_path, 
    orient="records", 
    lines=True, 
    date_format="iso", 
    date_unit="us"
)

print(f"Wrote {sample_path} with {sample.num_rows} rows")

Wrote trips_sample.json with 100000 rows


- 2.2 Explore and query
Using `mongosh`, run the following queries:
1. How many documents are in the collection?
2. Find all trips with more than 4 passengers.
3. Find the average `tip_amount` grouped by `payment_type`.
4. Find the top 5 longest trips by `trip_distance`.

Queried from terminal with mongosh.

1.How many documents are in the collection?

taxi> db.trips.countDocuments()
100000

2. Find all trips with more than 4 passengers.

taxi> db.trips.find({passenger_count: {$gt:4}}).count()
2089

3. Find the average `tip_amount` grouped by `payment_type`.

taxi> db.trips.aggregate([
...   {
...     $group: {
...       _id: "$payment_type",
...       avg_tip: { $avg: "$tip_amount" } 
...     }
...   },
...   {
...     $sort: { _id: 1 }
...   }
... ])

[
  { _id: 1, avg_tip: 4.705123567618947 },
  { _id: 2, avg_tip: 0.001203177139634347 },
  { _id: 3, avg_tip: 0 },
  { _id: 4, avg_tip: 0.055960544856740256 }
]

4. Find the top 5 longest trips by `trip_distance`.

taxi> db.trips.aggregate([{$sort: {"trip_distance": -1}}, {$project: {trip_distance:1}}, {$limit: 5}])

[
  { _id: ObjectId('69b8482a9e8963bb9c911c4f'), trip_distance: 971.8 },
  { _id: ObjectId('69b8482a9e8963bb9c911c4c'), trip_distance: 964.6 },
  { _id: ObjectId('69b8482a9e8963bb9c910fa3'), trip_distance: 233.25 },
  { _id: ObjectId('69b8482a9e8963bb9c90fbd3'), trip_distance: 101.28 },
  { _id: ObjectId('69b8482a9e8963bb9c911d0c'), trip_distance: 97.12 }
]

- 2.3 Reflection
In your report, answer: MongoDB stores each trip as a JSON-like document. For this
kind of tabular, analytical data (millions of rows, aggregation queries), what are
the disadvantages of using MongoDB compared to Parquet + DuckDB?
  
Already included in short PDF report.

# 3. Format Comparison (25%)

- 3.1. Read and combine the data
  
Read the three monthly Parquet files and combine them into a single Arrow Table.
Hint: `pyarrow.concat_tables()` can combine multiple tables.



In [5]:
months = ["01", "02", "03"]
files = [f"yellow_tripdata_2024-{m}.parquet" for m in months]
combined_table = pa.concat_tables([pq.read_table(f) for f in files], promote_options='permissive')
print(f"Combined table: {combined_table.num_rows:,} rows.\n")

Combined table: 9,554,778 rows.



- 3.2 Export to other formats
  
Export the combined dataset to:
- CSV (full dataset)
- JSON Lines (a sample of 100,000 rows. A full export would be too large)

In [6]:
df = combined_table.to_pandas()

In [ ]:
csv_file = "combined_taxi_data.csv"
df.to_csv(csv_file, index=False) 
print(f"Exported {csv_file}.")

In [ ]:
jsonl_file = "trips_sample.json" 
df_sample = df.head(100000) 
df_sample.to_json(jsonl_file, orient="records", lines=True, date_format="iso")
print(f"Exported {jsonl_file}.\n")

- 3.3. Fill in a table with File Sizes and ratio with respect to parquet for each format.

In [ ]:
parquet_size_mb = sum(os.path.getsize(f) for f in files) / (1024 * 1024)
csv_size_mb = os.path.getsize(csv_file) / (1024 * 1024)
jsonl_sample_size_mb = os.path.getsize(jsonl_file) / (1024 * 1024)

# Extrapolate: (Sample Size / 100,000) * Total Rows
total_rows = len(df)
jsonl_extrapolated_mb = (jsonl_sample_size_mb / 100000) * total_rows

size_data = [
    ["Parquet Original", parquet_size_mb, parquet_size_mb / parquet_size_mb],
    ["CSV", csv_size_mb, csv_size_mb / parquet_size_mb],
    ["JSON Lines (Extrapolated)", jsonl_extrapolated_mb, jsonl_extrapolated_mb / parquet_size_mb]
]
size_df = pd.DataFrame(size_data, columns=["Format", "Size (MB)", "Ratio vs Parquet"])
print(size_df.to_string(index=False))
print("\n")

- 3.4 Compare read times

Measure how long it takes to read the full dataset from:
- The original Parquet files
- The CSV file you createdm

In [ ]:
# Time Parquet Read
start = time.time()
pa.concat_tables([pq.read_table(f) for f in files]) 
pq_time = time.time() - start
print(f"Read Parquet time: {pq_time:.4f}s")

In [ ]:
# Time CSV Read
start = time.time()
temp_csv = pd.read_csv(csv_file) 
csv_time = time.time() - start
del temp_csv # Delete immediately
gc.collect()
print(f"Read CSV time: {csv_time:.4f}s\n")

- 3.5 Compression comparison

Write the combined dataset as a single Parquet file with different compression
algorithms: `snappy`, `gzip`, `zstd`, and `none`.
For each one, record:
- File size
- Write time
- Read time
Present your results in a table and write 2-3 sentences about which compression
you would recommend and why -> Specified in shor report.

In [ ]:
compressions = ['snappy', 'gzip', 'zstd', 'none']
comp_results = []

for comp in compressions:
    out_file = f"compressed_{comp}.parquet"
    
    # Write
    t0 = time.time()
    pq.write_table(combined_table, out_file, compression=comp)
    t_write = time.time() - t0
    
    # Read
    t0 = time.time()
    pq.read_table(out_file)
    t_read = time.time() - t0
    
    size = os.path.getsize(out_file) / (1024 * 1024)
    comp_results.append({'Compression': comp, 'Size (MB)': size, 'Write Time (s)': t_write, 'Read Time (s)': t_read})

comp_df = pd.DataFrame(comp_results)
print(comp_df.to_string(index=False))

- 3.6 Column pruning

Choose an analytical question that only needs 2-3 columns (for example: "average
tip amount per payment type"). Measure the read time when:
- Reading all columns
- Reading only the columns you need
How much faster is column pruning?

In [ ]:
cols_needed = ['PULocationID']
test_file = "compressed_none.parquet"

# Full Read
start = time.time()
pq.read_table(test_file)
time_all = time.time() - start

# Pruned Read
start = time.time()
pq.read_table(test_file, columns=cols_needed)
time_pruned = time.time() - start

print(f"Question: What are the top 5 pickup locations?")
print(f"Read ALL columns time: {time_all:.4f}s")
print(f"Read PRUNED columns (1/19) time: {time_pruned:.4f}s")
print(f"Speedup: {time_all / time_pruned:.2f}x")

# Part 4: Partitioning Strategy (25%)

- 4.1 Choose a partition column

The taxi data has a `tpep_pickup_datetime` column (timestamp of pickup). Extract the month from this column and write the dataset as partitioned Parquet, partitioned by month

In [ ]:
df['pickup_month'] = df['tpep_pickup_datetime'].dt.month
table_with_month = pa.Table.from_pandas(df)

partition_path = "taxi_partitioned_month"
ds.write_dataset(table_with_month, partition_path, format="parquet",
                 partitioning=["pickup_month"], 
                 existing_data_behavior="overwrite_or_ignore")


- 4.2 Inspect the partitioned output

- How many partition directories were created?
- What are the file sizes in each partition?
- Are the partitions roughly balanced?

In [ ]:
partitions = [p for p in os.listdir(partition_path) if os.path.isdir(os.path.join(partition_path, p))]
print(f"Created {len(partitions)} partition directories.")

partition_sizes = {}
for p in partitions:
    p_path = os.path.join(partition_path, p)
    size_mb = sum(os.path.getsize(os.path.join(p_path, f)) for f in os.listdir(p_path)) / (1024*1024)
    partition_sizes[p] = size_mb
    print(f"  Directory {p:20} | Size: {size_mb:.2f} MB")

# Balance Check
max_s = max(partition_sizes.values())
min_s = min(partition_sizes.values())
print(f"Balance check: Max={max_s:.2f}MB, Min={min_s:.2f}MB. Diff={max_s - min_s:.2f}MB")


- 4.3 Partition pruning
  
Write a query that filters for a single month (e.g., January 2024). Measure the
read time:
- Reading from the partitioned dataset (with pruning)
- Reading from the non-partitioned single file (filtering in memory)
How much faster is partition pruning?

In [ ]:
# Read from partitioned dataset (Pruning)
start = time.time()
dataset = ds.dataset(partition_path, format="parquet", partitioning=["pickup_month"])
# Pruning happens here: PyArrow only opens the 'pickup_month=1' folder
table_pruned = dataset.to_table(filter=(ds.field("pickup_month") == 1))
time_pruning = time.time() - start

# Read from single file (No Pruning - In Memory Filter)
start = time.time()
single_file = "compressed_none.parquet"
table_full = pq.read_table(single_file)
# We have to read everything then filter the rows
table_filtered = table_full.filter(pc.equal(pc.month(table_full['tpep_pickup_datetime']), 1))
time_memory = time.time() - start

print(f"Read with Partition Pruning: {time_pruning:.4f}s")
print(f"Read with Memory Filtering:  {time_memory:.4f}s")
print(f"Speedup: {time_memory / time_pruning:.2f}x")


- 4.4. Over-partitioning

Now partition by day (extract year, month, and day). Inspect the result:
- How many partition directories were created?
- What are the file sizes?
- Is this a good partitioning strategy? Why or why not?

In [ ]:
df['pickup_year'] = df['tpep_pickup_datetime'].dt.year
df['pickup_day'] = df['tpep_pickup_datetime'].dt.day
table_ymd = pa.Table.from_pandas(df)

ymd_path = "taxi_partitioned_ymd"
ds.write_dataset(table_ymd, ymd_path, format="parquet",
                 partitioning=["pickup_year", "pickup_month", "pickup_day"], 
                 existing_data_behavior="overwrite_or_ignore")

# Count all subdirectories
dir_count = sum(len(dirnames) for _, dirnames, _ in os.walk(ymd_path))
print(f"Total directories created: {dir_count}")

# Check one random file size
first_leaf = next(os.walk(ymd_path))[1][0] # first year folder

print("Observation: This strategy creates too many small files. This is 'Over-partitioning'.")


- 4.5 Metadata inspection
  
Use either `pq.read_metadata()` or DuckDB's `parquet_metadata()` to inspect one
of the Parquet files you created. Report:
- How many row groups does it have?
- What are the min/max statistics for a numeric column?
- How could a query engine use these statistics to skip data?

In [ ]:
meta = pq.read_metadata("compressed_none.parquet")
print(f"Number of Row Groups: {meta.num_row_groups}")

col_name = 'trip_distance'
col_idx = table_with_month.column_names.index(col_name)
rg_meta = meta.row_group(0).column(col_idx)

print(f"Statistics for '{col_name}' in Row Group 0:")
print(f"  Min: {rg_meta.statistics.min}")
print(f"  Max: {rg_meta.statistics.max}")

print("\nReflection: A query engine uses these statistics for 'Predicate Pushdown'.")
print("If a query asks for trips > 100 miles, and a Row Group's Max is 50, the engine skips that entire group.")